# Design Experimental
Este notebook descreve o design experimental utilizado no desenvolvimento do trabalho. A geração do arquivo `csv` contendo a tabela de experimentos foi criada a partir das bibliotecas `pyDOE3`, `pandas` e `numpy`.

In [1]:
import numpy as np
import pandas as pd
from pyDOE3 import fullfact

## Fatores variados

Os fatores variados no experimento são:

- **n_gpus**: número de GPUs utilizadas na inferência:
    - `1`: baseline single-GPU, sem particionamento;
    - `2` e `4`: configurações particionadas em múltiplas máquinas.
- **no_pcad**: tipo do nó PCAD utilizado, com características distintas de VRAM:
    - *tupi*: nós com 1× NVIDIA GeForce RTX 4090 (24 GB) cada;
    - *poti*: nós com 1× NVIDIA GeForce RTX 4070 (12 GB) cada.
- **estrategia**: estratégia de particionamento do modelo entre as GPUs:
    - *TP* (Tensor Parallelism): cada camada é dividida horizontalmente entre as GPUs, que sincronizam os resultados via all-reduce após cada camada;
    - *PP* (Pipeline Parallelism): as camadas são distribuídas verticalmente entre as GPUs, que trabalham em sequência passando ativações de um estágio ao próximo;
    - *none*: modelo carregado por completo em uma única GPU, sem particionamento.
- **prompt**: perfil de tamanho da entrada e da saída em tokens:
    - *short*: 128 tokens de entrada e 128 tokens de saída;
    - *long*: 1024 tokens de entrada e 512 tokens de saída.

Como `tupi` e `poti` têm uma única GPU por nó, toda configuração com `n_gpus > 1` envolve múltiplas máquinas (cross-node).

In [2]:
fator_n_gpus = [1, 2, 4]
fator_no_pcad = ["tupi", "poti"]
fator_estrategia = ["TP", "PP", "none"]
fator_prompt = ["short", "long"]

REPLICATIONS = 1
SEED = 42

niveis = [
    len(fator_n_gpus),
    len(fator_no_pcad),
    len(fator_estrategia),
    len(fator_prompt),
]
print(niveis)

[3, 2, 3, 2]


## Geração do fatorial completo
A função `fullfact` realiza a fatoração completa e retorna índices das listas pra cada nível.

In [3]:
design_indices = fullfact(niveis).astype(int)
print(design_indices[:10])

[[0 0 0 0]
 [1 0 0 0]
 [2 0 0 0]
 [0 1 0 0]
 [1 1 0 0]
 [2 1 0 0]
 [0 0 1 0]
 [1 0 1 0]
 [2 0 1 0]
 [0 1 1 0]]


Os índices retornados pelo `fullfact` são convertidos para os valores reais de cada fator, gerando o `DataFrame` inicial com todas as combinações possíveis do fatorial completo. Nesta etapa, ainda não há filtro algum: as 36 linhas representam o produto cartesiano dos níveis de `n_gpus`, `no_pcad`, `estrategia` e `prompt`.

In [4]:
df = pd.DataFrame({
    "n_gpus": [fator_n_gpus[i] for i in design_indices[:, 0]],
    "no_pcad": [fator_no_pcad[i] for i in design_indices[:, 1]],
    "estrategia": [fator_estrategia[i] for i in design_indices[:, 2]],
    "prompt": [fator_prompt[i] for i in design_indices[:, 3]],
})

print(f"Fatorial completo: {len(df)} combinações")

Fatorial completo: 36 combinações


## Filtros de restrições estruturais

Nem todas as combinações do fatorial completo são fisicamente realizáveis ou fazem sentido experimental. Os filtros aplicados a seguir eliminam células inválidas:

- **`n_gpus = 1`**: só faz sentido com `estrategia = none` (não há o que particionar com uma única GPU). Além disso, restringimos o baseline single-GPU ao `tupi` (24 GB), já que o Qwen 2.5 7B em FP16 (~15 GB) não cabe em um único `poti` (12 GB).
- **`n_gpus = 2`**: a estratégia `none` não se aplica, pois é obrigatório particionar o modelo. Esta configuração é executada tanto em `tupi` quanto em `poti`, permitindo comparar o impacto da VRAM disponível por nó na mesma topologia multi-nó.
- **`n_gpus = 4`**: a estratégia `none` também não se aplica. Restringimos esta configuração ao `poti`, já que não dispomos de 4 nós `tupi` para o experimento.

Como `tupi` e `poti` têm apenas 1 GPU por nó, todas as configurações com `n_gpus > 1` necessariamente envolvem múltiplas máquinas (cross-node).

Após essa etapa, restam apenas as células experimentalmente viáveis.

In [ ]:
# n_gpus = 1: só cabe em tupi (poti tem 12 GB, modelo precisa de ~15 GB) e estrategia=none
df = df[~((df["n_gpus"] == 1) & (df["estrategia"] != "none"))]
df = df[~((df["n_gpus"] == 1) & (df["no_pcad"] != "tupi"))]

# n_gpus > 1: estrategia deve ser TP ou PP (não none)
df = df[~((df["n_gpus"] > 1) & (df["estrategia"] == "none"))]

# n_gpus = 4: só roda em poti (não dispomos de 4 nós tupi)
df = df[~((df["n_gpus"] == 4) & (df["no_pcad"] == "tupi"))]

df = df.reset_index(drop=True)
print(f"Após filtros estruturais: {len(df)} células únicas")

## Replicação

Cada célula experimental válida é replicada `REPLICATIONS` vezes. As réplicas permitem estimar a variabilidade entre execuções e calcular intervalos de confiança sobre as métricas de desempenho coletadas.

In [6]:
df_replicated = pd.concat(
    [df.assign(replicacao=i) for i in range(1, REPLICATIONS + 1)],
    ignore_index=True,
)
print(f"Após {REPLICATIONS} réplicas: {len(df_replicated)} runs totais")

Após 1 réplicas: 10 runs totais


## Ordenação para execução

As runs são agrupadas por nó PCAD (`tupi` ou `poti`) e por número de GPUs, de modo que jobs destinados ao mesmo tipo de partição fiquem contíguos e evitem alternar topologias / restartar o cluster Ray. Dentro de cada agrupamento, a ordem das runs é completamente aleatorizada (com `SEED`), randomizando estratégias, prompts e réplicas para controlar efeitos sistemáticos ao longo do tempo de execução.

In [7]:
df_replicated = (
    df_replicated
    .sample(frac=1, random_state=SEED)
    .sort_values(by=["no_pcad", "n_gpus"], kind="stable")
    .reset_index(drop=True)
)

## Colunas derivadas

A partir dos fatores principais são geradas colunas auxiliares que parametrizam a execução de cada run:

- **`nodes`**: número de máquinas envolvidas. Como `tupi` e `poti` têm uma única GPU por nó, `nodes` coincide com `n_gpus`.
- **`gpus_per_node`**: número de GPUs disponíveis por nó na partição alvo (`1` em `tupi` e `poti`).
- **`tp`** e **`pp`**: graus de paralelismo tensorial e de pipeline, ajustados conforme a estratégia escolhida (apenas o eixo correspondente recebe `n_gpus`; o outro fica em 1).
- **`input_tokens`** e **`output_tokens`**: tamanhos do prompt de entrada e da geração, mapeados a partir do perfil `short` (128/128) ou `long` (1024/512).

In [8]:
# tupi e poti têm 1 GPU por nó; nodes coincide com n_gpus
df_replicated["gpus_per_node"] = 1
df_replicated["nodes"] = df_replicated["n_gpus"]

df_replicated["tp"] = df_replicated.apply(
    lambda r: r["n_gpus"] if r["estrategia"] == "TP" else 1, axis=1
)
df_replicated["pp"] = df_replicated.apply(
    lambda r: r["n_gpus"] if r["estrategia"] == "PP" else 1, axis=1
)

df_replicated["input_tokens"]  = df_replicated["prompt"].map({"short": 128,  "long": 1024})
df_replicated["output_tokens"] = df_replicated["prompt"].map({"short": 128,  "long": 512})

## Identificador único por run (`Order`)

Cada execução recebe um identificador único que combina os valores dos fatores e o número da réplica (`r1`, `r2`, ...). Esse rótulo serve para nomear logs, arquivos de saída e correlacionar resultados com a ordem real de execução.

In [9]:
df_replicated["Order"] = [
    f"N{r.n_gpus}_{r.no_pcad}_{r.estrategia}_{r.prompt}_r{r.replicacao}"
    for r in df_replicated.itertuples()
]

## Exportação

Apenas as colunas usadas como parâmetros para rodar os jobs são exportadas para `projeto_experimental.csv` (`Order`, `n_gpus`, `no_pcad`, `nodes`, `gpus_per_node`, `tp`, `pp`, `input_tokens`, `output_tokens`). Os fatores `estrategia`, `prompt` e `replicacao` ficam codificados no `Order` e podem ser recuperados depois — não precisam estar no CSV de execução. O resumo por célula segue usando o `DataFrame` completo em memória para conferência.

In [10]:
colunas_export = [
    "Order",
    "n_gpus",
    "no_pcad", "nodes", "gpus_per_node", "tp", "pp",
    "input_tokens", "output_tokens",
]

output_csv = f"projeto_experimental.csv"
df_replicated[colunas_export].to_csv(output_csv, index=False)

print(f"\nSalvo em: {output_csv}")
print(f"\nPrimeiras 10 linhas:")
print(df_replicated[colunas_export].head(10).to_string(index=False))

print(f"\nResumo por célula:")
resumo = df_replicated.groupby(
    ["n_gpus", "no_pcad", "estrategia", "prompt"], observed=True
).size().reset_index(name="n_runs")
print(resumo.to_string(index=False))


Salvo em: projeto_experimental.csv

Primeiras 10 linhas:
                Order  n_gpus no_pcad  nodes  gpus_per_node  tp  pp  input_tokens  output_tokens
   N2_poti_TP_long_r1       2    poti      2              1   2   1          1024            512
  N2_poti_TP_short_r1       2    poti      2              1   2   1           128            128
   N2_poti_PP_long_r1       2    poti      2              1   1   2          1024            512
  N2_poti_PP_short_r1       2    poti      2              1   1   2           128            128
   N4_poti_PP_long_r1       4    poti      4              1   1   4          1024            512
  N4_poti_TP_short_r1       4    poti      4              1   4   1           128            128
  N4_poti_PP_short_r1       4    poti      4              1   1   4           128            128
   N4_poti_TP_long_r1       4    poti      4              1   4   1          1024            512
 N1_tupi_none_long_r1       1    tupi      1              1   1   1  